<a href="https://colab.research.google.com/github/adhammkhaled/Pattern-Recognition-Labs/blob/main/Style%20Transfer/cnn_style_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 03

  Style Transfer with Multi-Input Conditioning:
In this assignment, you will implement a neural style transfer pipeline that takes two input images
and a style threshold:
- A content image, such as a photograph of a city or a person
- A style image, such as a painting by Van Gogh or Picasso
- The style threshold is a scalar value that determines the degree to which the style is
transferred onto the content image


Your model will generate a new image that preserves the content of the first image while adopting
the style of the second, modulated by the threshold value. To understand the underlying principles
of neural style transfer, you are expected to read the foundational paper on this topic https://openaccess.thecvf.com/content_cvpr_2016/papers/Gatys_Image_Style_Transfer_CVPR_2016_paper.pdf

## **Part One**

In [1]:
!pip install kagglehub

In [6]:
import kagglehub
import os
import random
from sklearn.model_selection import train_test_split
from PIL import Image
import shutil

### 1.1. Downloading

In [28]:
# downloading WikiArt style dataset
style_path = kagglehub.dataset_download("steubk/wikiart")
print("WikiArt style images saved to:", style_path)

# downloading content image dataset
content_path = kagglehub.dataset_download("duttadebadri/image-classification")
print("Content images saved to:", content_path)

content_path = os.path.join(content_path, "images", "images")



WikiArt style images saved to: /kaggle/input/wikiart
Content images saved to: /kaggle/input/image-classification


In [25]:
print(os.listdir(content_path))
print("\n")
print(os.listdir(style_path))

['food and d rinks', 'travel and  adventure', 'art and culture', 'architecure']


['Pop_Art', 'Minimalism', 'Color_Field_Painting', 'Mannerism_Late_Renaissance', 'Symbolism', 'Impressionism', 'Contemporary_Realism', 'High_Renaissance', 'Fauvism', 'Rococo', 'Early_Renaissance', 'Naive_Art_Primitivism', 'Pointillism', 'classes.csv', 'Cubism', 'Synthetic_Cubism', 'Action_painting', 'Abstract_Expressionism', 'New_Realism', 'wclasses.csv', 'Baroque', 'Analytical_Cubism', 'Expressionism', 'Romanticism', 'Northern_Renaissance', 'Ukiyo_e', 'Post_Impressionism', 'Art_Nouveau_Modern', 'Realism']


In [33]:
content_image_count = 0
for root, _, files in os.walk(content_path):
    content_image_count += len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f"Number of content images: {content_image_count}")




Number of content images: 35093


In [30]:
style_image_count = 0
for subfolder in os.listdir(style_path):
    subfolder_path = os.path.join(style_path, subfolder)
    if os.path.isdir(subfolder_path):
        style_image_count += len([
            f for f in os.listdir(subfolder_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
print(f"Number of style images: {style_image_count}")

Number of style images: 81444


### 1.2. Splitting

In [35]:
# 1. Collect content and style image paths
def get_image_paths(root_dir, limit=None):
    image_paths = []
    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(root, file))
    if limit:
        image_paths = image_paths[:limit]  # or use random.sample(...) for randomness
    return image_paths

content_images = get_image_paths(content_path, limit=5000)
style_images = get_image_paths(style_path, limit=5000)

print(f"Selected content images: {len(content_images)}")
print(f"Selected style images: {len(style_images)}")

# 2. Split into train/val/test
def split_data(images):
    train, temp = train_test_split(images, test_size=0.3, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)
    return train, val, test

content_train, content_val, content_test = split_data(content_images)
style_train, style_val, style_test = split_data(style_images)

# 3. Copy images to new folders
def copy_images(images, base_dir, label):
    for img_path in images:
        filename = os.path.basename(img_path)
        dest_dir = os.path.join(base_dir, label)
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(img_path, os.path.join(dest_dir, filename))

base_output = "/content/dataset_split"

# Create folders like /content/dataset_split/content/train, etc.
for split, data in zip(
    ['train', 'val', 'test'],
    [content_train, content_val, content_test]
):
    copy_images(data, os.path.join(base_output, 'content'), split)

for split, data in zip(
    ['train', 'val', 'test'],
    [style_train, style_val, style_test]
):
    copy_images(data, os.path.join(base_output, 'style'), split)

print("Done, dataset is split and stored under /content/dataset_split")


Selected content images: 5000
Selected style images: 5000
✅ Done! Dataset is split and stored under /content/dataset_split


In [36]:
def count_images_in_split(base_path, label):
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(base_path, label, split)
        num_images = len([
            f for f in os.listdir(split_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        print(f"{label.capitalize()} - {split}: {num_images} images")

# Check content
count_images_in_split("/content/dataset_split", "content")

# Check style
count_images_in_split("/content/dataset_split", "style")


Content - train: 3500 images
Content - val: 750 images
Content - test: 750 images
Style - train: 3478 images
Style - val: 750 images
Style - test: 748 images


## **Part Two : Data Preprocessing**